In [ ]:
from google.colab import drive
from pathlib import Path
from sklearn.neighbors import BallTree

import unicodedata
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

drive.mount("/content/drive")


def resolve_korean_path(path):
    target = Path(path)
    current = Path(target.anchor)

    for part in target.parts[1:]:
        direct = current / part

        if direct.exists():
            current = direct
            continue

        matches = [
            item for item in current.iterdir()
            if unicodedata.normalize("NFC", item.name)
            == unicodedata.normalize("NFC", part)
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"경로 확인 필요: {current / part}"
            )

        current = matches[0]

    return current


base = resolve_korean_path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2"
)

source_path = resolve_korean_path(
    str(base / "인허가_master.csv")
)

OUT = resolve_korean_path(
    str(base / "인허가_파생지표_202301_202604")
)

SOURCE_DIR = OUT / "snapshots"
NEW_DIR = OUT / "같은업태추가"
NEW_SNAPSHOT_DIR = NEW_DIR / "snapshots"

NEW_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

needed = [
    "점포이력ID",
    "업태구분명",
    "lon",
    "lat",
]

for encoding in ["utf-8-sig", "cp949"]:
    try:
        attributes = pd.read_csv(
            source_path,
            encoding=encoding,
            usecols=needed,
            dtype="string",
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("CSV 인코딩을 확인하세요.")

if (
    attributes["점포이력ID"].isna().any()
    or attributes["점포이력ID"].duplicated().any()
):
    raise ValueError("원본 점포이력ID의 결측·중복을 확인하세요.")

# 공백만 정리하며, 서로 다른 업태 명칭을 임의로 합치지 않음
attributes["업태구분명"] = (
    attributes["업태구분명"]
    .str.strip()
    .replace("", pd.NA)
)

for col in ["lon", "lat"]:
    attributes[col] = pd.to_numeric(
        attributes[col],
        errors="coerce",
    ).astype(float)

attributes["좌표사용가능"] = (
    np.isfinite(attributes["lon"])
    & np.isfinite(attributes["lat"])
    & attributes["lon"].between(-180, 180)
    & attributes["lat"].between(-90, 90)
    & ~(
        attributes["lon"].eq(0)
        & attributes["lat"].eq(0)
    )
)

# 기존 컬럼과 충돌하지 않도록 좌표는 보조 이름 사용
attributes = attributes.rename(columns={
    "lon": "_경도",
    "lat": "_위도",
})

months = pd.date_range(
    "2023-01-01",
    "2026-04-01",
    freq="MS",
)

source_files = [
    SOURCE_DIR / f"snapshot_{t:%Y%m}.parquet"
    for t in months
]

missing = [str(p) for p in source_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "월별 파일 누락:\n" + "\n".join(missing)
    )

print("원본 업태 정보:", len(attributes), "개")
print("업태 결측:", attributes["업태구분명"].isna().sum(), "개")
print("저장 위치:", NEW_DIR)

In [ ]:
EARTH_RADIUS_M = 6_371_008.8

new_features = [
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

reports = []

for i, (t, path) in enumerate(
    zip(months, source_files),
    start=1,
):
    x = pd.read_parquet(path)

    required = [
        "점포이력ID",
        "행정동코드",
        "구군",
        "업종명",
    ]

    missing = set(required) - set(x.columns)
    if missing:
        raise ValueError(
            f"{path.name}: 필수 열 누락 {missing}"
        )

    if x["점포이력ID"].duplicated().any():
        raise ValueError(f"{path.name}: 점포번호 중복")

    original_columns = x.columns.tolist()
    original_rows = len(x)

    # 재실행해도 같은 컬럼이 중복되지 않도록 처리
    x = x.drop(
        columns=["업태구분명"] + new_features,
        errors="ignore",
    )

    x = x.merge(
        attributes,
        on="점포이력ID",
        how="left",
        validate="one_to_one",
        indicator="_업태연결",
    )

    if x["_업태연결"].ne("both").any():
        raise ValueError(
            f"{path.name}: 원본과 연결되지 않는 점포가 있습니다."
        )

    # 빈 업종·업태는 같은 그룹으로 묶지 않음
    valid_type = (
        x["업종명"].notna()
        & x["업종명"].astype("string").str.strip().ne("")
        & x["업태구분명"].notna()
    ).fillna(False)

    for col in new_features:
        x[col] = pd.Series(
            pd.NA,
            index=x.index,
            dtype="Int32",
        )

    # 1. 같은 행정동·업종·업태, 자신 제외
    counts = (
        x.loc[valid_type]
        .groupby(
            ["행정동코드", "업종명", "업태구분명"],
            observed=True,
        )["점포이력ID"]
        .transform("size")
        - 1
    )

    x.loc[
        counts.index,
        "동네_같은업태점포수",
    ] = counts.astype("Int32")

    # 2. 같은 구군·업종·업태, 자신 제외
    counts = (
        x.loc[valid_type]
        .groupby(
            ["구군", "업종명", "업태구분명"],
            observed=True,
        )["점포이력ID"]
        .transform("size")
        - 1
    )

    x.loc[
        counts.index,
        "구군_같은업태점포수",
    ] = counts.astype("Int32")

    # 3. 반경 300m 같은 업종·업태, 자신 제외
    valid_spatial = (
        valid_type
        & x["좌표사용가능"].fillna(False)
    )

    for _, group in x.loc[valid_spatial].groupby(
        ["업종명", "업태구분명"],
        observed=True,
        sort=False,
    ):
        points = np.radians(
            group[["_위도", "_경도"]].to_numpy(dtype=float)
        )

        tree = BallTree(
            points,
            metric="haversine",
        )

        counts = tree.query_radius(
            points,
            r=300 / EARTH_RADIUS_M,
            count_only=True,
        ) - 1

        x.loc[
            group.index,
            "300미터내_같은업태점포수",
        ] = pd.array(counts, dtype="Int32")

    # 보조 컬럼 제거
    x = x.drop(columns=[
        "_경도",
        "_위도",
        "좌표사용가능",
        "_업태연결",
    ])

    assert len(x) == original_rows

    for col in new_features:
        assert x[col].dropna().ge(0).all()

    # 기존 변수는 그대로 보존됐는지 확인
    preserved = [
        c for c in original_columns
        if c not in new_features
        and c != "업태구분명"
    ]

    assert set(preserved).issubset(x.columns)

    output = (
        NEW_SNAPSHOT_DIR
        / f"snapshot_{t:%Y%m}.parquet"
    )

    x.to_parquet(
        output,
        index=False,
        compression="snappy",
    )

    report = {
        "기준월": t.strftime("%Y-%m"),
        "점포수": len(x),
        "업태결측수": int(
            x["업태구분명"].isna().sum()
        ),
    }

    for col in new_features:
        report[f"{col}_결측률"] = float(
            x[col].isna().mean()
        )

    reports.append(report)

    pd.DataFrame(reports).to_csv(
        NEW_DIR / "같은업태_계산요약.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[{i:02d}/40] {t:%Y-%m} | "
        f"{len(x):,}행 저장 완료",
        flush=True,
    )

    del x

display(pd.DataFrame(reports))

In [ ]:
paths = [
    NEW_SNAPSHOT_DIR / f"snapshot_{t:%Y%m}.parquet"
    for t in months
]

output_path = (
    NEW_DIR
    / "인허가_파생지표_전체_업태추가_202301_202604.parquet"
)

temp_path = output_path.with_suffix(".parquet.part")


def plain_schema(schema):
    # 범주형은 문자열 등 원래 값의 자료형으로 통일
    return pa.schema([
        pa.field(
            field.name,
            (
                field.type.value_type
                if pa.types.is_dictionary(field.type)
                else field.type
            ),
            nullable=True,
        )
        for field in schema
    ])


schemas = [
    plain_schema(pq.read_schema(path))
    for path in paths
]

for path, schema in zip(paths, schemas):
    if set(schema.names) != set(schemas[0].names):
        raise ValueError(
            f"월별 컬럼 구성이 다릅니다: {path.name}"
        )

schema = pa.unify_schemas(
    schemas,
    promote_options="permissive",
)

total_rows = 0

with pq.ParquetWriter(
    temp_path,
    schema,
    compression="snappy",
) as writer:
    for i, path in enumerate(paths, start=1):
        table = pq.read_table(path)

        table = table.cast(
            plain_schema(table.schema)
        )

        table = table.select(
            schema.names
        ).cast(schema)

        writer.write_table(table)
        total_rows += table.num_rows

        print(
            f"[{i:02d}/40] 전체 파일 합치는 중 | "
            f"누적 {total_rows:,}행",
            flush=True,
        )

        del table

assert (
    pq.ParquetFile(temp_path).metadata.num_rows
    == total_rows
)

temp_path.replace(output_path)

# 한 달만 읽어 확인
df_check = pd.read_parquet(
    output_path,
    filters=[("base_ym", "==", "2026-03")],
    columns=[
        "base_ym",
        "점포이력ID",
        "업종명",
        "업태구분명",
    ] + new_features,
)

display(df_check.head(20))

print("전체 파일 저장 완료:", output_path)
print(f"전체 행 수: {total_rows:,}")

In [ ]:
# 업태별 점포 수가 추가된 전체 파일 읽기
df = pd.read_parquet(output_path)

print("파일:", output_path)
print(f"전체 행 수: {len(df):,}")
print(f"전체 컬럼 수: {len(df.columns):,}")

# 업태별 점포 수 확인
check_columns = [
    "base_ym",
    "점포이력ID",
    "행정동코드",
    "업종명",
    "업태구분명",
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

display(df[check_columns].head(20))

# 2026년 3월 일반음식점만 확인
display(
    df.loc[
        df["base_ym"].eq("2026-03")
        & df["업종명"].eq("일반음식점"),
        check_columns,
    ].head(30)
)

In [ ]:
print(df.columns.tolist())

In [ ]:
# 컬럼명의 한글 숫자만 숫자로 변경
replacements = {
    "삼백미터": "300미터",
    "백미터": "100미터",
    "육개월": "6개월",
    "일년": "1년",
}

old_columns = df.columns.tolist()

def change_numbers(name):
    for korean, number in replacements.items():
        name = name.replace(korean, number)
    return name

new_columns = [change_numbers(col) for col in old_columns]

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경 후 컬럼명이 중복됩니다.")

df.columns = new_columns

print(df.columns.tolist())
display(df.head(10))

In [ ]:
df[df['업태구분명'] == "편의점"]

In [ ]:
# 1. 모델에 사용할 파생지표
feature_cols = [
    # 점포 특성
    "영업기간_개월",
    "영업기간_구간",
    "개업1년미만",
    "점포업종",
    "프랜차이즈여부",

    # 동네·동종 업종 환경
    "동네점포_영업기간중앙값_개월",
    "주변동종_폐업전평균영업기간_개월",
    "주변동종_신규점포_6개월생존율",
    "주변동종_프랜차이즈비율",
    "주변동종_6개월폐업률",
    "주변동종_6개월개업률",
    "주변동종_6개월점포증감",
    "동네_업종다양성",

    # 거리 기반 환경
    "300미터내_동종점포수",
    "100미터내_6개월폐업수",

    # 같은 업태 점포 수
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

target_col = "앞으로6개월내_폐업여부"
key_cols = ["기준월", "점포번호"]

# 필수 컬럼 확인
required = key_cols + feature_cols + [target_col]
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(
        "현재 df에서 아래 컬럼을 찾을 수 없습니다:\n"
        + "\n".join(missing)
    )


# 2. 시각화용 데이터프레임
viz_candidates = [
    # 점포·지역·업태 정보
    "기준월",
    "점포번호",
    "구군",
    "행정동코드",
    "행정동명",
    "법정동",
    "업종명",
    "업태구분명",
    "개업일",
    "경도",
    "위도",
    "좌표유효여부",

    # 주변 점포 수
    "동네_동종점포수",
    "구군_동종점포수",
    "동네동종_6개월전점포수",
    "동네동종_6개월개업수",
    "동네동종_6개월폐업수",

    # 집계 신뢰도 확인용
    "동네_영업기간계산점포수",
    "동네동종_일년폐업수",
    "동네동종_생존율계산점포수",

    # 파생지표와 정답
    *feature_cols,
    target_col,
]

# 파일에 실제 존재하는 컬럼만 선택하고 중복 제거
viz_cols = list(dict.fromkeys(
    col for col in viz_candidates if col in df.columns
))

# 업종명과 점포업종이 같은 내용이면 시각화용에서는 하나만 유지
if (
    "업종명" in viz_cols
    and "점포업종" in viz_cols
    and df["업종명"].astype("string").equals(
        df["점포업종"].astype("string")
    )
):
    viz_cols.remove("점포업종")

df_viz = df.loc[:, viz_cols].copy()


# 3. 모델링용 데이터프레임
# 기준월·점포번호는 추적 및 시간 분할용이며 모델 입력에서는 제외
model_cols = key_cols + feature_cols + [target_col]
df_model = df.loc[:, model_cols].copy()

print("시각화용 df_viz:", df_viz.shape)
print("모델링용 df_model:", df_model.shape)
print("모델 입력 피처 수:", len(feature_cols))

print("\n시각화용")
display(df_viz.head())

print("\n모델링용")
display(df_model.head())

print(
    "\n정답이 있는 행:",
    f"{df_model[target_col].notna().sum():,}",
)

In [ ]:
from google.colab import drive
from pathlib import Path
import unicodedata
import pandas as pd

drive.mount("/content/drive")

root = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

# 한글 컬럼으로 저장한 전체 파일을 우선 사용
candidate_names = [
    "인허가_파생지표_업태추가_한글컬럼_전체_202301_202604.parquet",
    "인허가_파생지표_전체_업태추가_202301_202604.parquet",
]

parquet_path = None

for name in candidate_names:
    matches = [
        path for path in root.rglob("*.parquet")
        if unicodedata.normalize("NFC", path.name) == name
    ]

    if len(matches) > 1:
        raise ValueError(
            "동일한 이름의 파일이 여러 개입니다:\n"
            + "\n".join(map(str, matches))
        )

    if matches:
        parquet_path = matches[0]
        break

if parquet_path is None:
    raise FileNotFoundError("업태 추가 전체 Parquet 파일을 찾지 못했습니다.")

output_path = parquet_path
print("불러올 파일:", parquet_path)

In [ ]:
df = pd.read_parquet(parquet_path)

print("데이터 크기:", df.shape)
print(df.columns.tolist())
display(df.head())

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display

import re
import unicodedata
import numpy as np
import pandas as pd

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

def normalize_name(name):
    return unicodedata.normalize("NFC", name)

# 파일 목록을 한 번만 탐색
parquet_files = list(ROOT.rglob("*.parquet"))
csv_files = list(ROOT.rglob("*.csv"))

# 업태 추가 원본을 우선 사용하고, 없으면 한글 컬럼 저장본 사용
candidate_names = [
    "인허가_파생지표_전체_업태추가_202301_202604.parquet",
    "인허가_파생지표_업태추가_한글컬럼_전체_202301_202604.parquet",
]

parquet_path = None

for filename in candidate_names:
    matches = [
        path for path in parquet_files
        if normalize_name(path.name) == filename
    ]

    if len(matches) > 1:
        raise ValueError(
            "동일한 결과 파일이 여러 개입니다:\n"
            + "\n".join(map(str, matches))
        )

    if matches:
        parquet_path = matches[0]
        break

if parquet_path is None:
    raise FileNotFoundError(
        "업태별 점포 수가 추가된 전체 Parquet 파일을 찾지 못했습니다."
    )

master_matches = [
    path for path in csv_files
    if normalize_name(path.name) == "인허가_master.csv"
]

if len(master_matches) != 1:
    raise ValueError(
        f"인허가_master.csv가 {len(master_matches)}개 발견됐습니다:\n"
        + "\n".join(map(str, master_matches))
    )

master_path = master_matches[0]

print("파생지표 파일:", parquet_path)
print("인허가 원본:", master_path)

In [ ]:
df = pd.read_parquet(parquet_path)

print(f"행 수: {len(df):,}")
print(f"컬럼 수: {len(df.columns):,}")

display(df.head(5))

In [ ]:
# (영문 이름, 최종 한글 이름, 이전 한글 이름들)
definitions = [
    ("base_ym", "기준월", []),
    ("점포이력ID", "점포번호", ["점포이력식별자"]),
    ("통합점포ID", "통합점포번호", ["통합점포식별자"]),
    ("점포ID", "인허가번호", ["점포식별자"]),

    ("open_s", "개업일", []),
    ("close_s", "폐업일", []),
    ("lon", "경도", []),
    ("lat", "위도", []),
    ("coord_valid", "좌표유효여부", []),

    ("age_m", "영업기간_개월", ["업력개월"]),
    ("age_band", "영업기간_구간", ["업력구간"]),
    ("is_new", "개업1년미만", ["신규점포여부"]),
    ("biz_type", "점포업종", ["세부업종"]),
    ("is_franchise", "프랜차이즈여부", ["프랜차이즈"]),

    (
        "dong_age_med",
        "동네점포_영업기간중앙값_개월",
        ["행정동업력중앙값"],
    ),
    (
        "closed_life_avg",
        "주변동종_폐업전평균영업기간_개월",
        ["폐업점포평균영업개월"],
    ),
    (
        "surv_6m",
        "주변동종_신규점포_6개월생존율",
        ["개업코호트6개월생존율", "신규점포_6개월생존율"],
    ),
    (
        "fr_share",
        "주변동종_프랜차이즈비율",
        ["동종프랜차이즈비중"],
    ),
    (
        "close_rate_6m",
        "주변동종_6개월폐업률",
        ["최근6개월폐업률"],
    ),
    (
        "open_rate_6m",
        "주변동종_6개월개업률",
        ["최근6개월개업률"],
    ),
    (
        "net_6m",
        "주변동종_6개월점포증감",
        ["최근6개월점포순증"],
    ),
    (
        "biz_entropy",
        "동네_업종다양성",
        ["행정동업종다양성"],
    ),

    (
        "same_300m",
        "300미터내_동종점포수",
        ["반경300미터동종영업점수"],
    ),
    (
        "close_100m_6m",
        "100미터내_6개월폐업수",
        ["반경100미터최근6개월폐업점수"],
    ),

    (
        "label_close_6m",
        "앞으로6개월내_폐업여부",
        ["향후6개월폐업여부"],
    ),
    (
        "n_dong_age",
        "동네_영업기간계산점포수",
        ["행정동업력집계점포수"],
    ),
    (
        "n_hk",
        "동네_동종점포수",
        ["행정동동종영업점수"],
    ),
    (
        "n_gk",
        "구군_동종점포수",
        ["구군동종영업점수"],
    ),
    (
        "n_entropy",
        "동네_다양성계산점포수",
        ["업종다양성집계점포수"],
    ),
]

# 보정 전 값
raw_definitions = [
    (
        "fr_share",
        "프랜차이즈비율",
        "동종프랜차이즈비중보정전",
    ),
    (
        "close_rate_6m",
        "6개월폐업률",
        "최근6개월폐업률보정전",
    ),
    (
        "open_rate_6m",
        "6개월개업률",
        "최근6개월개업률보정전",
    ),
    (
        "net_6m",
        "6개월점포증감",
        "최근6개월점포순증보정전",
    ),
    (
        "closed_life_avg",
        "폐업전平均영업기간_개월".replace("平均", "평균"),
        "폐업점포평균영업개월보정전",
    ),
    (
        "surv_6m",
        "신규점포_6개월생존율",
        "개업코호트6개월생존율보정전",
    ),
]

# 집계 표본 수
count_definitions = [
    (
        "n_fr_{level}",
        "프랜차이즈확인점포수",
        "동종프랜차이즈여부관측점포수",
    ),
    (
        "n_start_{level}k",
        "6개월전점포수",
        "동종6개월전영업점수",
    ),
    (
        "n_open_{level}k",
        "6개월개업수",
        "동종최근6개월개업점수",
    ),
    (
        "n_close_{level}k",
        "6개월폐업수",
        "동종최근6개월폐업점수",
    ),
    (
        "n_closed_life_{level}",
        "1년폐업수",
        "동종최근1년폐업점수",
    ),
    (
        "n_cohort_{level}",
        "생존율계산점포수",
        "동종생존율집계개업점수",
    ),
]

for level, old_region, region in [
    ("h", "행정동", "동네"),
    ("g", "구군", "구군"),
]:
    for prefix, meaning, old_suffix in raw_definitions:
        aliases = [f"{old_region}{old_suffix}"]

        # 이전 쉬운 이름도 지원
        if prefix == "closed_life_avg":
            aliases.append(
                f"{region}동종_폐업전평균영업기간_원래값"
            )
        elif prefix == "surv_6m":
            aliases.append(
                f"{region}동종_신규점포생존율_원래값"
            )

        definitions.append((
            f"{prefix}_{level}_raw",
            f"{region}동종_{meaning}_원래값",
            aliases,
        ))

    for pattern, meaning, old_suffix in count_definitions:
        definitions.append((
            pattern.format(level=level),
            f"{region}동종_{meaning}",
            [f"{old_region}{old_suffix}"],
        ))


def normalize_numbers(name):
    replacements = {
        "삼백미터": "300미터",
        "백미터": "100미터",
        "육개월": "6개월",
        "일년": "1년",
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    return name


column_map = {}

for english, korean, aliases in definitions:
    for name in [english, korean] + aliases:
        column_map[normalize_numbers(name)] = korean

old_columns = df.columns.tolist()

new_columns = [
    column_map.get(
        normalize_numbers(col),
        normalize_numbers(col),
    )
    for col in old_columns
]

# 업종__로 시작하는 기존 컬럼은 보존
unmapped = [
    col for col in new_columns
    if re.search(r"[A-Za-z]", col)
    and not col.startswith("업종__")
]

if unmapped:
    raise ValueError(
        f"한글 이름을 추가로 지정해야 하는 컬럼: {unmapped}"
    )

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경 후 컬럼명이 중복됩니다.")

df.columns = new_columns

column_name_table = pd.DataFrame({
    "기존컬럼명": old_columns,
    "한글컬럼명": new_columns,
})

print("한글 컬럼명 변경 완료")
display(column_name_table)

In [ ]:
for encoding in ["utf-8-sig", "cp949"]:
    try:
        header = pd.read_csv(
            master_path,
            encoding=encoding,
            nrows=0,
        )

        industry_cols = [
            col for col in header.columns
            if col.startswith("업종__")
        ]

        if not industry_cols:
            raise ValueError(
                "원본에서 업종__로 시작하는 컬럼을 찾지 못했습니다."
            )

        industry = pd.read_csv(
            master_path,
            encoding=encoding,
            usecols=["점포이력ID"] + industry_cols,
            dtype="string",
        )
        break

    except UnicodeDecodeError:
        continue
else:
    raise ValueError("원본 CSV 인코딩을 확인하세요.")

industry = industry.rename(
    columns={"점포이력ID": "점포번호"}
)

if (
    industry["점포번호"].isna().any()
    or industry["점포번호"].duplicated().any()
):
    raise ValueError("원본 점포번호에 결측 또는 중복이 있습니다.")

for col in industry_cols:
    values = pd.to_numeric(
        industry[col],
        errors="raise",
    )

    if values.isna().any() or not values.isin([0, 1]).all():
        raise ValueError(
            f"{col}: 0·1 이외 값 또는 결측이 있습니다."
        )

    industry[col] = values.astype("int8")

# 연결 키 자료형 통일
industry["점포번호"] = industry["점포번호"].astype("string")
df["점포번호"] = df["점포번호"].astype("string")

original_rows = len(df)

# 이미 있으면 원본 값으로 다시 연결
df = df.drop(
    columns=industry_cols,
    errors="ignore",
)

df = df.merge(
    industry,
    on="점포번호",
    how="left",
    validate="many_to_one",
    indicator="_연결상태",
)

if df["_연결상태"].ne("both").any():
    raise ValueError(
        "원본 업종 정보와 연결되지 않은 점포가 있습니다."
    )

df = df.drop(columns="_연결상태")

assert len(df) == original_rows

print(f"업종별 0·1 컬럼 복원 완료: {len(industry_cols)}개")
print("복원 후 전체 크기:", df.shape)

display(
    df[["기준월", "점포번호"] + industry_cols[:10]]
    .head(10)
)